In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
#!pip install nltk
# nltk.download('punkt_tab')

## Tokenize 
# 1. sent_tokenize
import nltk
from nltk.tokenize import sent_tokenize
data = """Hello, My name is Soumya Jaiswal. I'm not so good boy. By the way i am learning NLP now! The only thing currently i need."""
sent_tokenize(data)

['Hello, My name is Soumya Jaiswal.',
 "I'm not so good boy.",
 'By the way i am learning NLP now!',
 'The only thing currently i need.']

In [ ]:
# 2. word_tokenize
from nltk.tokenize import word_tokenize
data = """Hello, My name is Soumya Jaiswal. I'm not so good boy. By the way i am learning NLP now! The only thing currently i need."""
word_tokenize(data)[:10]

['Hello', ',', 'My', 'name', 'is', 'Soumya', 'Jaiswal', '.', 'I', "'m"]

In [ ]:



# 3. wordpunct_tokenize
from nltk.tokenize import wordpunct_tokenize
data = """Hello, My name is Soumya Jaiswal. I'm not so good boy. By the way i am learning NLP now! The only thing currently i need."""
wordpunct_tokenize(data)[:10]

['Hello', ',', 'My', 'name', 'is', 'Soumya', 'Jaiswal', '.', 'I', "'"]

In [ ]:
## Stemming
## 1.PorterStemmer
data = "The boys are playing football and running very fast towards the playground."
from nltk.stem import PorterStemmer
stemming = PorterStemmer()
ans = []
for x in wordpunct_tokenize(data):
    ans.append(stemming.stem(x))
" ".join(ans)

'the boy are play footbal and run veri fast toward the playground .'

In [ ]:
## 2.RegExpStemmer
data = "The boys are playing football and running very fast towards the playground."
from nltk.stem import RegexpStemmer
stemming = RegexpStemmer("ing$|s$|es$able$")
ans = []
for x in wordpunct_tokenize(data):
    ans.append(stemming.stem(x))
" ".join(ans)

'The boy are play football and runn very fast toward the playground .'

In [ ]:
## 3.SnowballStemmer
data = "The boys are playing football and running very fast towards the playground."
from nltk.stem import SnowballStemmer
stemming = SnowballStemmer('english')
ans = []
for x in wordpunct_tokenize(data):
    ans.append(stemming.stem(x))
" ".join(ans)

'the boy are play footbal and run veri fast toward the playground .'

In [ ]:
# Lemmatization                                                            nltk.download('wordnet')
data = "The boys are playing football and running very fast towards the playground."
from nltk.stem import WordNetLemmatizer
lemmitizer = WordNetLemmatizer()
ans = []
for x in wordpunct_tokenize(data):
    ans.append(lemmitizer.lemmatize(x, pos='v'))
" ".join(ans)

'The boys be play football and run very fast towards the playground .'

In [ ]:
# stopwords   -  unnecessary  pronoun  is/am are remover
 
data = """Hello, My name is Soumya Jaiswal. I'm not so good boy. By the way i am learning NLP now! The only thing currently i need."""
from nltk.corpus import stopwords
sentences = nltk.sent_tokenize(data)
stopwords = set(stopwords.words("english"))
print(sentences)
c = 0
for sen in sentences:
    words = nltk.wordpunct_tokenize(sen)
    words = [stemming.stem(word).lower() for word in words if word.lower() not in stopwords]
    sentences[c] = " ".join(words)
    c+=1
print(sentences)

['Hello, My name is Soumya Jaiswal.', "I'm not so good boy.", 'By the way i am learning NLP now!', 'The only thing currently i need.']
['hello , name soumya jaiswal .', "' good boy .", 'way learn nlp !', 'thing current need .']


In [ ]:
# stopwords   -  unnecessary  pronoun  is/am are remover
 
data = """Hello, My name is Soumya Jaiswal. I'm not so good boy. By the way i am learning NLP now! The only thing currently i need."""
from nltk.corpus import stopwords
sentences = nltk.sent_tokenize(data)
stopwords = set(stopwords.words("english"))
print(sentences)
c = 0
tag_elems = []
for sen in sentences:
    words = nltk.wordpunct_tokenize(sen)
    words = [stemming.stem(word).lower() for word in words if word.lower() not in stopwords]
    tag_elems.extend([x for x in nltk.pos_tag(words) ]) 
    print(tag_elems[c])
    sentences[c] = " ".join(words)
    c+=1
print(sentences)
nltk.ne_chunk(tag_elems).draw()

['Hello, My name is Soumya Jaiswal.', "I'm not so good boy.", 'By the way i am learning NLP now!', 'The only thing currently i need.']
('hello', 'NN')
(',', ',')
('name', 'NN')
('soumya', 'NN')
['hello , name soumya jaiswal .', "' good boy .", 'way learn nlp !', 'thing current need .']


# Training Sms Classifier

In [ ]:
# One hot encoding by making cols from words assign it [1, 1, 0, 0, 0,] -> 1 binary && 2 no of time it came
# bad if we choose it 

df = pd.read_csv("spam.csv", encoding="latin-1")
df = df.drop(["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"], axis=1)

In [ ]:


df = df.rename({"v1":"is_spam", "v2":"msg"}, axis=1)

In [ ]:
def is_spam(s):
    if "ham" in s.lower():
        return 0
    return 1
df["is_spam"] = df["is_spam"].apply(is_spam)

In [ ]:
import re
corpus = []
review=0
for i in range(len(df)):
    review = re.sub("[^a-zA-Z]", " ", df["msg"][i])
    review = review.lower()
    review = review.split()
    words = [lemmitizer.lemmatize(word).lower() for word in review if word.lower() not in stopwords]
    corpus.append(" ".join(words))

In [ ]:
df["msg_corpus"] = corpus

In [ ]:
from sklearn.model_selection import train_test_split
X_trainn, X_testt, y_train, y_test = train_test_split(
    df["msg_corpus"],
    df["is_spam"],
    test_size=0.2,
    random_state=42
)

In [ ]:
print(df.isnull().sum())

is_spam       0
msg           0
msg_corpus    0
dtype: int64


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(
    max_features=1000,
    ngram_range=(1, 3) # tells to make combination from words as seperate features
)

X_train = cv.fit_transform(X_trainn)
X_test = cv.transform(X_testt)

In [ ]:
X_train=X_train.toarray()
X_test=X_test.toarray()

In [ ]:
[x for x in cv.vocabulary_.keys()][:6]

['thing', 'change', 'want', 'im', 'leaving', 'ok']

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

model = MultinomialNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9730700179533214
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       957
           1       0.91      0.90      0.90       157

    accuracy                           0.97      1114
   macro avg       0.95      0.94      0.94      1114
weighted avg       0.97      0.97      0.97      1114



In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

model = KNeighborsClassifier(
    n_neighbors=5,
    metric="minkowski",
    p=2          # Euclidean distance
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9353680430879713
              precision    recall  f1-score   support

           0       0.93      1.00      0.96       957
           1       1.00      0.54      0.70       157

    accuracy                           0.94      1114
   macro avg       0.97      0.77      0.83      1114
weighted avg       0.94      0.94      0.93      1114



In [ ]:
## TFIDF(better than bagging)
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1, 3)
)

X_train = tfidf.fit_transform(X_trainn)
X_test = tfidf.transform(X_testt)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

model = MultinomialNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9748653500897666
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       957
           1       0.97      0.85      0.90       157

    accuracy                           0.97      1114
   macro avg       0.97      0.92      0.95      1114
weighted avg       0.97      0.97      0.97      1114



In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

model = KNeighborsClassifier(
    n_neighbors=5,
    metric="minkowski",
    p=2          # Euclidean distance
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9236983842010772
              precision    recall  f1-score   support

           0       0.92      1.00      0.96       957
           1       0.97      0.47      0.64       157

    accuracy                           0.92      1114
   macro avg       0.95      0.73      0.80      1114
weighted avg       0.93      0.92      0.91      1114



In [ ]:
!pip install gensim

  Using cached smart_open-8.0.1-py3-none-any.whl.metadata (24 kB)
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
    --------------------------------------- 0.5/24.4 MB 524.3 kB/s eta 0:00:46
    --------------------------------------- 0.5/24.4 MB 524.3 kB/s eta 0:00:46
   - -------------------------------------- 0.8/24.4 MB 532.3 kB/s eta 0:00:45
   - -------------------------------------- 1.0/24.4 MB 699.0 kB/s eta 0:00:34
   -- ------------------------------------- 1.3/24.4 MB 798.6 kB/s eta 0:00:29
   -- ------------------------------------- 1.6/24.4 MB 864.6 kB/s eta 0:00:27
   --- ------------------------------------ 1.8/24.4 MB 890.6 kB/s eta 0:00:26
   --- ------------------------


[notice] A new release of pip is available: 25.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from gensim.models import Word2Vec

sentences = [text.split() for text in df["msg_corpus"]]

w2v = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["msg_corpus"],
    df["is_spam"],
    test_size=0.2,
    random_state=42
)
import numpy as np

def average_word2vec(sentence, model):
    words = sentence.split()

    vectors = [
        model.wv[word]
        for word in words
        if word in model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)
X_train_vec = np.array([
    average_word2vec(sentence, w2v)
    for sentence in X_train
])

X_test_vec = np.array([
    average_word2vec(sentence, w2v)
    for sentence in X_test
])

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = LogisticRegression(max_iter=1000)

model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)

print(accuracy_score(y_test, y_pred))

0.8581687612208259


In [35]:
import gensim.downloader as api
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Load pretrained Google News Word2Vec (300D)
google_model = api.load("word2vec-google-news-300")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    df["msg_corpus"],
    df["is_spam"],
    test_size=0.2,
    random_state=42
)

# Average Word2Vec
def average_word2vec(sentence):
    words = sentence.split()

    vectors = [
        google_model[word]
        for word in words
        if word in google_model
    ]

    if len(vectors) == 0:
        return np.zeros(google_model.vector_size)

    return np.mean(vectors, axis=0)

# Convert text to vectors
X_train_vec = np.array([
    average_word2vec(sentence)
    for sentence in X_train
])

X_test_vec = np.array([
    average_word2vec(sentence)
    for sentence in X_test
])

# Train classifier
model = LogisticRegression(max_iter=1000)

model.fit(X_train_vec, y_train)

# Predict
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

[===================-------------------------------] 38.3% 636.9/1662.8MB downloaded

ContentTooShortError: <urlopen error retrieval incomplete: got only 667877376 out of 1743563840 bytes>

In [36]:
king_vec = wv["king"]

print(king_vec.shape)

NameError: name 'wv' is not defined

In [37]:
wv.most_similar("cricket")

NameError: name 'wv' is not defined

In [38]:
wv.most_similar("cricket", topn=5)

NameError: name 'wv' is not defined

In [39]:
king_vec = wv["king"]


NameError: name 'wv' is not defined

In [40]:
wv.similar_by_vector(king_vec)

NameError: name 'wv' is not defined

In [41]:
wv.most_similar([king_vec])

NameError: name 'wv' is not defined

In [42]:
wv.most_similar(
    positive=["king", "woman"],
    negative=["man"],
    topn=5
)

NameError: name 'wv' is not defined